# Goal and Work Summary

**Goal**: Build and evaluate a patch-based semantic segmentation pipeline to detect defects in green rough oak planks (Background, BlackRot, Knot, Stain). This notebook covers the full data workflow, patch training, and both patch-level and full-image predictions with quantitative metrics and visual diagnostics.

## Workflow Stages and Rationale

**Stage 1 — Data audit and class normalization**
- Scan the raw line-scan dataset to quantify images and available class masks.
- Normalize inconsistent defect names to a single taxonomy.
- Use custom helpers in `notebooks/functions` for repeatability.

**Why this choice**: Stable training and honest evaluation require consistent labels and a reproducible audit.

**Stage 2 — Filtering rare classes**
- Move extremely low-frequency classes out of the training pool using a fixed threshold.

**Why this choice**: Rare classes can destabilize optimization and skew loss scaling in segmentation.

**Stage 3 — Reorganization and validation**
- Reorganize the dataset into aligned image/mask pairs.
- Validate integrity (missing masks, bad samples) and isolate invalid items.

**Why this choice**: Clean structure prevents silent misalignment during training and evaluation.

**Stage 4 — Mask composition and splits**
- Merge per-class masks into a single multi-class mask per image.
- Persist train/val/test splits to JSON for reproducible experiments.

**Why this choice**: Multi-class masks match model outputs and keep evaluation consistent across runs.

**Stage 5 — Patch dataset + sampling strategy**
- Define a patch dataset with Albumentations transforms (size, patches per image, optional stride for tiling).
- Use positive-patch sampling to control the fraction of defect-containing patches.
- Normalize with ImageNet stats to match the pretrained ResNet34 encoder.

**Why this choice**: Patch training improves class balance, fits GPU memory, and focuses on local defect cues while preserving encoder priors.

**Stage 6 — Model training and checkpointing**
- Train a UNet with ResNet34 encoder using CrossEntropy loss.
- Track loss and IoU in TensorBoard; save checkpoints for reproducibility.

**Why this choice**: UNet is a strong segmentation baseline; ResNet34 balances capacity and speed.

**Stage 7 — Best-model selection and patch-level evaluation**
- Select the best checkpoint by validation IoU and reload it for test inference.
- Report mean IoU, ROC curves (one-vs-rest), F1, accuracy, and percentage confusion matrices computed over patches.

**Why this choice**: Validation-based selection is fair, and multiple metrics reveal different failure modes.

**Stage 8 — Full-image predictions from patch model**
- Run full-image inference by tiling each large image into patches with a fixed stride.
- Perform model inference per patch, then stitch predictions back into the original image space.
- When patches overlap, aggregate class probabilities (softmax) before taking the final argmax label map.
- Save visual overlays and compute full-image metrics on reconstructed masks.

**Why this choice**: Patch-trained models can still produce full-image segmentations by sliding-window inference, enabling realistic deployment-style evaluation.

## Custom Functions Used

We rely on project-specific helpers (in `notebooks/functions` and this notebook) to standardize data processing:
- `analyze_dataset`, `reorganize_dataset`, `validate_dataset`, `move_invalid_samples`
- `create_combined_masks`, `create_train_val_split`
- `rename_defect_files`, `filter_classes_by_mask_count`
- `create_dataloaders_patch`, `PatchDataset`

These encapsulate repeatable steps and document the intended data workflow.

## References (Libraries)

| Library | Purpose | Notes | Link |
| --- | --- | --- | --- |
| PyTorch | Training, inference, dataloaders | Core deep learning framework and CUDA support | https://pytorch.org/ |
| segmentation_models_pytorch | UNet + encoder zoo | Provides ResNet34-UNet and pretrained encoders | https://github.com/qubvel/segmentation_models.pytorch |
| Albumentations | Augmentations + normalization | Fast, flexible image transforms for segmentation | https://albumentations.ai/ |
| albumentations.pytorch | Tensor conversion | Provides `ToTensorV2` for patch transforms | https://albumentations.ai/docs/api_reference/pytorch/ |
| OpenCV | Image utilities | Used for file I/O and preprocessing support | https://opencv.org/ |
| NumPy | Array ops | Efficient tensor/array manipulation | https://numpy.org/ |
| Pillow | Image reading | Handles TIFF/PNG image loading | https://python-pillow.org/ |
| scikit-learn | Metrics | ROC, F1, accuracy, confusion matrix | https://scikit-learn.org/ |
| Matplotlib | Plotting | Curves and confusion matrix visualization | https://matplotlib.org/ |
| TensorBoard | Logging | Visual tracking of training metrics | https://www.tensorflow.org/tensorboard |


In [1]:
# Prepare Dataset
from functions.data_pipeline import prepare_dataset
import segmentation_models_pytorch as smp
from torch.utils.tensorboard import SummaryWriter

data_root, class_mapping = prepare_dataset(
    mask_threshold=200,     # minimum masks to keep a defect class
    val_split=0.16,
    test_split=0.2,
)
NUM_CLASSES = len(class_mapping) + 1  # +1 for Background

PREPARE_DATASET — SKIPPED (data already prepared)
  Organized data: ../data/organized-data
  Classes: {'BlackRot': 1, 'Knot': 2, 'Stain': 3}
  (pass force=True to re-run the full pipeline)


In [ ]:
# ──────────────────────────────────────────────
# Configuration - change these knobs as needed
# ──────────────────────────────────────────────
import torch

GPU_ID = 0  # choose 0 or 1

if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)           # sets the current CUDA device
    DEVICE = torch.device(f"cuda:{GPU_ID}") # use this for .to(DEVICE)
    print(f"Using GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

device = DEVICE  # used by training / dataloaders cells below


### Positive patch sampling

During training, a percentage of patches are forced to contain defects to fight class imbalance.

- `pos_fraction`: fraction of train patches that should be positive (e.g., `0.5` = 50%).
- `min_pos_pixels`: minimum defect pixels required to count a patch as positive.
- `max_pos_tries`: number of random attempts to find a positive patch before falling back.
- `pos_classes`: optional list of class ids to treat as positive; `None` means any non-zero class.

Example usage:

```python
train_loader, val_loader = create_dataloaders_patch(
    data_root="../data/organized-data",
    batch_size=32,
    patch_size=224,
    patches_per_image=4,
    stride=112,
    pos_fraction=0.5,
    min_pos_pixels=50,
    max_pos_tries=30,
    pos_classes=[1, 2, 3],
)
```

If you see too many empty patches, increase `pos_fraction` or `min_pos_pixels`. If training becomes unstable, reduce `pos_fraction` or increase `max_pos_tries`.

In [2]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import json
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class PatchSegmentationDataset(Dataset):
    """
    Patch-based segmentation dataset:
      - train: returns random patches (patches_per_image per image)
      - val/test: returns tiled patches covering each image (stride controls overlap)
    """
    def __init__(
        self,
        data_root,
        split='train',
        transform=None,
        patch_size=512,
        patches_per_image=4,
        stride=None,
        # positive sampling controls
        pos_fraction=0.5,          # fraction of train patches forced to be positive
        min_pos_pixels=25,         # minimum positive pixels in a patch to be "positive"
        max_pos_tries=20,          # attempts to find a positive patch before fallback
        pos_classes=None,          # optional list of class ids to treat as positive
    ):
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform
        self.patch_size = int(patch_size)
        self.patches_per_image = int(patches_per_image)
        self.stride = stride if stride is not None else self.patch_size
        self.pos_fraction = float(pos_fraction)
        self.min_pos_pixels = int(min_pos_pixels)
        self.max_pos_tries = int(max_pos_tries)
        self.pos_classes = pos_classes  # None means any non-zero is positive

        split_file = self.data_root / "split.json"
        with open(split_file, 'r') as f:
            split_info = json.load(f)
        self.image_ids = split_info[split]

        self.images_dir = self.data_root / "images"
        self.masks_dir = self.data_root / "combined_masks"

        # For non-train splits, precompute tiles (list of (img_id, y, x))
        if self.split != 'train':
            self.tiles = []
            for img_id in self.image_ids:
                img_path = self.images_dir / f"{img_id}_Col.tif"
                img = np.array(Image.open(img_path).convert('RGB'))
                H, W = img.shape[:2]
                stride = self.stride
                for y in range(0, max(1, H - self.patch_size + 1), stride):
                    for x in range(0, max(1, W - self.patch_size + 1), stride):
                        self.tiles.append((img_id, y, x))
                # ensure coverage of right/bottom edges
                if (H - self.patch_size) % stride != 0:
                    y = max(0, H - self.patch_size)
                    for x in range(0, max(1, W - self.patch_size + 1), stride):
                        self.tiles.append((img_id, y, x))
                if (W - self.patch_size) % stride != 0:
                    x = max(0, W - self.patch_size)
                    for y in range(0, max(1, H - self.patch_size + 1), stride):
                        self.tiles.append((img_id, y, x))
                    if (H - self.patch_size) % stride != 0:
                        self.tiles.append((img_id, max(0, H - self.patch_size), max(0, W - self.patch_size)))

        print(f"Loaded {split} dataset: {len(self.image_ids)} images; patch_size={self.patch_size}")

    def __len__(self):
        if self.split == 'train':
            return len(self.image_ids) * self.patches_per_image
        return len(self.tiles)

    def _load_image_and_mask(self, img_id):
        img_path = self.images_dir / f"{img_id}_Col.tif"
        mask_path = self.masks_dir / f"{img_id}_mask.png"
        image = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path))
        return image, mask

    def _pad_if_needed(self, arr, target_h, target_w, is_mask=False):
        h, w = arr.shape[:2]
        pad_h = max(0, target_h - h)
        pad_w = max(0, target_w - w)
        if pad_h == 0 and pad_w == 0:
            return arr
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        if is_mask:
            return cv2.copyMakeBorder(
                arr, top, bottom, left, right,
                borderType=cv2.BORDER_CONSTANT, value=0
            )
        return cv2.copyMakeBorder(arr, top, bottom, left, right, borderType=cv2.BORDER_REFLECT)

    def _is_positive(self, mask_patch):
        if self.pos_classes is None:
            return (mask_patch > 0).sum() >= self.min_pos_pixels
        # only count specified class ids as positive
        return np.isin(mask_patch, self.pos_classes).sum() >= self.min_pos_pixels

    def __getitem__(self, idx):
        if self.split == 'train':
            img_idx = idx // self.patches_per_image
            img_id = self.image_ids[img_idx]
            image, mask = self._load_image_and_mask(img_id)

            H, W = image.shape[:2]
            if H < self.patch_size or W < self.patch_size:
                image = self._pad_if_needed(image, self.patch_size, self.patch_size, is_mask=False)
                mask = self._pad_if_needed(mask, self.patch_size, self.patch_size, is_mask=True)
                H, W = image.shape[:2]

            # positive sampling
            want_pos = np.random.rand() < self.pos_fraction
            y = np.random.randint(0, H - self.patch_size + 1)
            x = np.random.randint(0, W - self.patch_size + 1)

            if want_pos:
                found = False
                for _ in range(self.max_pos_tries):
                    y = np.random.randint(0, H - self.patch_size + 1)
                    x = np.random.randint(0, W - self.patch_size + 1)
                    candidate = mask[y:y+self.patch_size, x:x+self.patch_size]
                    if self._is_positive(candidate):
                        found = True
                        break
                # if not found, fall back to last sampled location

            img_patch = image[y:y+self.patch_size, x:x+self.patch_size]
            mask_patch = mask[y:y+self.patch_size, x:x+self.patch_size]
        else:
            img_id, y, x = self.tiles[idx]
            image, mask = self._load_image_and_mask(img_id)
            H, W = image.shape[:2]
            if H < self.patch_size or W < self.patch_size:
                image = self._pad_if_needed(image, self.patch_size, self.patch_size, is_mask=False)
                mask = self._pad_if_needed(mask, self.patch_size, self.patch_size, is_mask=True)
            img_patch = image[y:y+self.patch_size, x:x+self.patch_size]
            mask_patch = mask[y:y+self.patch_size, x:x+self.patch_size]

        if self.transform:
            transformed = self.transform(image=img_patch, mask=mask_patch)
            img_patch = transformed['image']
            mask_patch = transformed['mask']

        return img_patch, mask_patch.long()

# Patch-aware augmentations (do NOT include Resize)
def get_training_augmentation_patches():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
        A.OneOf([A.RandomBrightnessContrast(p=1), A.RandomGamma(p=1)], p=0.3),
        A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

def get_validation_augmentation_patches():
    return A.Compose([
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

def create_dataloaders_patch(
    data_root,
    batch_size=4,
    patch_size=224,
    patches_per_image=4,
    stride=None,
    num_workers=4,
    pos_fraction=0.5,
    min_pos_pixels=25,
    max_pos_tries=20,
    pos_classes=None,
):
    train_dataset = PatchSegmentationDataset(
        data_root=data_root, split='train',
        transform=get_training_augmentation_patches(),
        patch_size=patch_size, patches_per_image=patches_per_image, stride=stride,
        pos_fraction=pos_fraction, min_pos_pixels=min_pos_pixels, max_pos_tries=max_pos_tries,
        pos_classes=pos_classes
    )
    val_dataset = PatchSegmentationDataset(
        data_root=data_root, split='val',
        transform=get_validation_augmentation_patches(),
        patch_size=patch_size, patches_per_image=1, stride=stride
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

In [3]:
# Build patch dataloaders

data_root = "../data/organized-data"
patch_size = 224
stride = 112  # 50% overlap for val tiling
batch_size = 32

train_loader, val_loader = create_dataloaders_patch(
    data_root=data_root,
    batch_size=batch_size,
    patch_size=patch_size,
    patches_per_image=4,
    stride=stride,
    num_workers=4,
    pos_fraction=0.5,
    min_pos_pixels=50,
    max_pos_tries=30,
    pos_classes=[1, 2, 3],  # non-background classes
)

# Build model
num_classes = 4

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
).to(device)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Model classes:", num_classes)

/home/vscode/.local/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Loaded train dataset: 688 images; patch_size=224
Loaded val dataset: 132 images; patch_size=224
Train batches: 86
Val batches: 726
Model classes: 4


### Model Training

In [4]:
# CUDA device is selected in the Configuration cell (GPU_ID / DEVICE).

CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 3090


In [5]:
# Patch training loop (optimized for patch dataset)
from torch.cuda.amp import autocast, GradScaler
import torch.nn as nn
from torch.optim import Adam
from pathlib import Path
import json

# Setup training
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 50
checkpoint_dir = Path("../checkpoints/UNet_Patches")
checkpoint_dir.mkdir(exist_ok=True)
model_dir = Path("../models/UNet_Patches")
model_dir.mkdir(exist_ok=True)

# LR scheduler (val IoU-aware)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

# Early stopping
patience = 8
min_delta = 1e-4
patience_counter = 0

# Training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_iou": [],
    "lr": [],
}

best_val_iou = -1.0
scaler = GradScaler(enabled=(device.type == "cuda"))

# Create a TensorBoard writer
writer = SummaryWriter(log_dir="../logs/tensorboard/ResNet34_Unet_Patches")

print("Using device:", device)
print("Model device:", next(model.parameters()).device)


def compute_iou_batch(pred, target, num_classes):
    """Compute mean IoU across classes for one batch."""
    ious = []
    for cls in range(num_classes):
        pred_mask = pred == cls
        target_mask = target == cls
        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()
        iou = intersection / (union + 1e-6)
        ious.append(iou)
    return torch.stack(ious).mean().item()


def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch on random patches."""
    model.train()
    total_loss = 0.0

    for img_patch, mask_patch in train_loader:
        img_patch = img_patch.to(device, non_blocking=True)
        mask_patch = mask_patch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type == "cuda")):
            logits = model(img_patch)
            loss = criterion(logits, mask_patch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / max(1, len(train_loader))


def validate(model, val_loader, criterion, device, num_classes):
    """Validate on tiled patches."""
    model.eval()
    total_loss = 0.0
    total_iou = 0.0

    with torch.no_grad():
        for img_patch, mask_patch in val_loader:
            img_patch = img_patch.to(device, non_blocking=True)
            mask_patch = mask_patch.to(device, non_blocking=True)

            with autocast(enabled=(device.type == "cuda")):
                logits = model(img_patch)
                loss = criterion(logits, mask_patch)

            pred = torch.argmax(logits, dim=1)
            iou = compute_iou_batch(pred, mask_patch, num_classes)

            total_loss += loss.item()
            total_iou += iou

    return (
        total_loss / max(1, len(val_loader)),
        total_iou / max(1, len(val_loader)),
    )


# Training loop
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, criterion, device, num_classes)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)

    # Step LR scheduler on val IoU
    scheduler.step(val_iou)
    current_lr = optimizer.param_groups[0]["lr"]
    history["lr"].append(current_lr)

    print(
        f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | LR: {current_lr:.2e}"
    )

    # Log to TensorBoard
    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("IoU/Val", val_iou, epoch)
    writer.add_scalar("LR", current_lr, epoch)

    # Save best checkpoint by val IoU
    if val_iou > best_val_iou + min_delta:
        best_val_iou = val_iou
        patience_counter = 0
        best_path = model_dir / "model_best.pt"
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_iou": val_iou,
            },
            best_path,
        )
        print(f"  → Best checkpoint saved: {best_path}")
    else:
        patience_counter += 1

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        ckpt_path = checkpoint_dir / f"model_epoch_{epoch+1}.pt"
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_iou": val_iou,
            },
            ckpt_path,
        )
        print(f"  → Checkpoint saved: {ckpt_path}")

    # Early stopping
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1} (best Val IoU: {best_val_iou:.4f})")
        break

# Save final history
history_path = checkpoint_dir / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"\nTraining complete. History saved to {history_path}")


Using device: cuda
Model device: cuda:0


/tmp/ipykernel_60491/3105948430.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))
/tmp/ipykernel_60491/3105948430.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):
/tmp/ipykernel_60491/3105948430.py:101: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):


Epoch 1/50 | Train Loss: 0.7532 | Val Loss: 0.4256 | Val IoU: 0.2244 | LR: 1.00e-03
  → Best checkpoint saved: ../models/UNet_Patches/model_best.pt
Epoch 2/50 | Train Loss: 0.5237 | Val Loss: 0.3600 | Val IoU: 0.2500 | LR: 1.00e-03
  → Best checkpoint saved: ../models/UNet_Patches/model_best.pt
Epoch 3/50 | Train Loss: 0.5032 | Val Loss: 0.3365 | Val IoU: 0.2453 | LR: 1.00e-03
Epoch 4/50 | Train Loss: 0.4771 | Val Loss: 0.3116 | Val IoU: 0.2669 | LR: 1.00e-03
  → Best checkpoint saved: ../models/UNet_Patches/model_best.pt
Epoch 5/50 | Train Loss: 0.4750 | Val Loss: 0.5486 | Val IoU: 0.2424 | LR: 1.00e-03
  → Checkpoint saved: ../checkpoints/UNet_Patches/model_epoch_5.pt
Epoch 6/50 | Train Loss: 0.4782 | Val Loss: 0.3684 | Val IoU: 0.2522 | LR: 1.00e-03
Epoch 7/50 | Train Loss: 0.4583 | Val Loss: 0.3573 | Val IoU: 0.2454 | LR: 1.00e-03
Epoch 8/50 | Train Loss: 0.4516 | Val Loss: 0.3622 | Val IoU: 0.2897 | LR: 1.00e-03
  → Best checkpoint saved: ../models/UNet_Patches/model_best.pt
Epoch

KeyboardInterrupt: 

### Test Set Inference (Best Model)

Load the test loader, restore the best checkpoint, and run patch‑level inference.

In [6]:
def compute_iou_batch(pred, target, num_classes):
    """Compute mean IoU across classes for one batch."""
    ious = []
    for cls in range(num_classes):
        pred_mask = pred == cls
        target_mask = target == cls
        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()
        iou = intersection / (union + 1e-6)
        ious.append(iou)
    return torch.stack(ious).mean().item()

In [7]:
# Build test loader (tiled patches)

test_dataset = PatchSegmentationDataset(
    data_root=data_root,
    split="test",
    transform=get_validation_augmentation_patches(),
    patch_size=patch_size,
    patches_per_image=1,
    stride=stride,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

# Load best model
best_model_path = "../models/UNet_Patches/model_best.pt"
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=num_classes,
).to(device)

ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(
    f"Loaded best model: {best_model_path} "
    f"(epoch={ckpt.get('epoch')}, val_iou={ckpt.get('val_iou'):.4f})"
)

# Predict test set

total_iou = 0.0
num_batches = 0
total_correct = 0
total_pixels = 0

with torch.no_grad():
    for img_patch, mask_patch in test_loader:
        img_patch = img_patch.to(device, non_blocking=True)
        mask_patch = mask_patch.to(device, non_blocking=True)

        logits = model(img_patch)
        preds = torch.argmax(logits, dim=1)

        total_iou += compute_iou_batch(preds, mask_patch, num_classes)
        num_batches += 1

        total_correct += (preds == mask_patch).sum().item()
        total_pixels += mask_patch.numel()

mean_iou = total_iou / max(1, num_batches)
pixel_acc = total_correct / max(1, total_pixels)

print(f"Test mean IoU (patch‑level): {mean_iou:.4f}")
print(f"Test pixel accuracy: {pixel_acc:.4f}")

Loaded test dataset: 205 images; patch_size=224
Loaded best model: ../models/UNet_Patches/model_best.pt (epoch=22, val_iou=0.3100)
Test mean IoU (patch‑level): 0.3062
Test pixel accuracy: 0.9078


### Full‑Image Reconstruction Metrics

Reconstruct full‑image predictions from tiled patches and report pixel accuracy + IoU.

In [8]:
# Reconstruct full‑image predictions from tiled patches
# MEMORY-EFFICIENT: Process images one at a time instead of loading all at once

import numpy as np
from PIL import Image
from collections import defaultdict

# Get image dimensions WITHOUT loading full images (memory-efficient)
print("Getting image dimensions...")
image_dims = {}
for img_id in test_dataset.image_ids:
    img_path = test_dataset.images_dir / f"{img_id}_Col.tif"
    with Image.open(img_path) as img:
        H, W = img.size[1], img.size[0]  # PIL uses (width, height)
    if H < patch_size or W < patch_size:
        H = max(H, patch_size)
        W = max(W, patch_size)
    image_dims[img_id] = (H, W)

print(f"Found {len(image_dims)} images. Processing one at a time...")

# Group tiles by image_id to process images sequentially
tiles_by_image = defaultdict(list)
for idx, (img_id, y, x) in enumerate(test_dataset.tiles):
    tiles_by_image[img_id].append((idx, y, x))

# Accumulators for final metrics
intersection = np.zeros(num_classes, dtype=np.float64)
union = np.zeros(num_classes, dtype=np.float64)
correct_pixels = 0
all_pixels = 0
all_preds = []
all_targets = []
all_probs = []

model.eval()

# Process each image completely before moving to the next
for img_idx, img_id in enumerate(test_dataset.image_ids):
    if (img_idx + 1) % 10 == 0:
        print(f"Processing image {img_idx + 1}/{len(test_dataset.image_ids)}: {img_id}")
    
    H, W = image_dims[img_id]
    
    # Initialize arrays for THIS image only
    sum_logits = np.zeros((num_classes, H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)
    
    # Get all tiles for this image
    image_tiles = tiles_by_image[img_id]
    
    # Process all patches for this image
    with torch.no_grad():
        for tile_idx, (global_idx, y, x) in enumerate(image_tiles):
            # Get the patch from the dataset
            img_patch, _ = test_dataset[global_idx]
            img_patch = img_patch.unsqueeze(0).to(device, non_blocking=True)
            
            # Run inference
            logits = model(img_patch)  # [1, C, H, W]
            logits_cpu = logits[0].cpu().numpy()  # [C, H, W]
            
            # Accumulate logits
            sum_logits[:, y:y + patch_size, x:x + patch_size] += logits_cpu
            count_map[y:y + patch_size, x:x + patch_size] += 1.0
    
    # Normalize by count (handle overlapping patches)
    counts = count_map.copy()
    counts[counts == 0] = 1.0  # safety
    avg_logits = sum_logits / counts[np.newaxis, :, :]
    preds = np.argmax(avg_logits, axis=0)
    
    # Load mask for this image (only when needed)
    mask_path = test_dataset.masks_dir / f"{img_id}_mask.png"
    target = np.array(Image.open(mask_path))
    if target.shape[0] < patch_size or target.shape[1] < patch_size:
        target = test_dataset._pad_if_needed(target, patch_size, patch_size, is_mask=True)
    # Ensure dimensions match
    if target.shape[:2] != (H, W):
        if target.shape[0] > H or target.shape[1] > W:
            target = target[:H, :W]
        else:
            target = test_dataset._pad_if_needed(target, H, W, is_mask=True)
    
    # Compute probabilities for ROC
    probs = np.exp(avg_logits - avg_logits.max(axis=0, keepdims=True))
    probs = probs / probs.sum(axis=0, keepdims=True)
    
    # Accumulate metrics
    correct_pixels += (preds == target).sum()
    all_pixels += target.size
    
    for cls in range(num_classes):
        pred_mask = preds == cls
        target_mask = target == cls
        intersection[cls] += np.logical_and(pred_mask, target_mask).sum()
        union[cls] += np.logical_or(pred_mask, target_mask).sum()
    
    # Store for confusion matrix and ROC
    all_preds.append(preds.reshape(-1))
    all_targets.append(target.reshape(-1))
    all_probs.append(probs.transpose(1, 2, 0).reshape(-1, num_classes))
    
    # Clear arrays for this image (free memory)
    del sum_logits, count_map, avg_logits, preds, target, probs

print("Finished processing all images!")

# Compute final metrics (already accumulated per-image above)

iou_per_class = np.divide(
    intersection,
    union,
    out=np.zeros_like(intersection, dtype=np.float64),
    where=union != 0,
)
mean_iou_incl_bg = iou_per_class.mean()
mean_iou_excl_bg = iou_per_class[1:].mean() if num_classes > 1 else mean_iou_incl_bg
pixel_acc_full = correct_pixels / max(1, all_pixels)

print("Full‑image reconstruction metrics")
print(f"Pixel accuracy: {pixel_acc_full:.4f}")
print(f"Mean IoU (incl background): {mean_iou_incl_bg:.4f}")
print(f"Mean IoU (excl background): {mean_iou_excl_bg:.4f}")

# Class‑wise IoU + confusion matrix + F1 + ROC

from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_names = ["Background", "BlackRot", "Knot"]

# Concatenate all predictions (already collected per-image above)
all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
all_probs = np.concatenate(all_probs)

print("IoU per class:")
for i, iou in enumerate(iou_per_class):
    print(f"  {class_names[i]}: {iou:.4f}")

cm = confusion_matrix(all_targets, all_preds, labels=list(range(num_classes)))
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", values_format=".1%", colorbar=True, xticks_rotation=45
)
plt.title("Confusion Matrix (Percent of True Class)")
plt.tight_layout()
plt.show()

f1_per_class = f1_score(all_targets, all_preds, labels=list(range(num_classes)), average=None)
f1_macro = f1_score(all_targets, all_preds, labels=list(range(num_classes)), average="macro")

print("F1 per class:")
for i, f1 in enumerate(f1_per_class):
    print(f"  {class_names[i]}: {f1:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")

# ROC curves (subsampled to avoid OOM)
max_roc_samples = 300_000  # adjust down if still heavy
rng = np.random.default_rng(42)

n = all_targets.shape[0]
if n > max_roc_samples:
    idx = rng.choice(n, size=max_roc_samples, replace=False)
    y_true_sample = all_targets[idx]
    y_score_sample = all_probs[idx]
else:
    y_true_sample = all_targets
    y_score_sample = all_probs

plt.figure(figsize=(7, 5))
for c in range(num_classes):
    y_true_bin = (y_true_sample == c).astype(int)
    y_score = y_score_sample[:, c]

    if y_true_bin.sum() == 0:
        print(f"ROC skipped: {class_names[c]} (no positives in sample)")
        continue

    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[c]} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One‑vs‑Rest, Full‑Image Reconstruction, Sampled)")
plt.legend()
plt.tight_layout()
plt.show()

Getting image dimensions...
Found 205 images. Processing one at a time...
Processing image 10/205: 3-18-25_8.05.01.635_Bot
Processing image 20/205: 10-27-25_8.01.03.510_Bot
Processing image 30/205: 2-6-26_8.52.52.160_Top
Processing image 40/205: 11-3-25_8.05.17.481_Top
Processing image 50/205: 11-3-25_7.38.59.290_Bot
Processing image 60/205: 10-13-25_10.36.12.648_Top
Processing image 70/205: 1-30-26_8.40.53.366_Top
Processing image 80/205: 10-29-25_11.12.45.669_Top
Processing image 90/205: 2-6-26_9.12.26.158_Top
Processing image 100/205: 2-6-26_9.12.24.549_Bot
Processing image 110/205: 2-18-26_2.32.44.384_Top
Processing image 120/205: 2-6-26_8.52.01.900_Bot
Processing image 130/205: 10-13-25_10.41.54.142_Bot
Processing image 140/205: 2-6-26_8.54.08.729_Bot
Processing image 150/205: 8-11-25_12.13.52.355_Bot
Processing image 160/205: 10-27-25_8.05.32.995_Bot
Processing image 170/205: 11-5-25_9.37.57.753_Bot
Processing image 180/205: 2-6-26_8.51.57.013_Top
Processing image 190/205: 11-3-2

IndexError: list index out of range

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.onnx
import numpy as np

# Figure out project root (works when running from notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


# ── EBI-Compliant Export Wrapper ──────────────────────
class EBIExportWrapper(nn.Module):
    """
    Wraps a trained segmentation model for EBI-compliant ONNX export.

    The ONNX graph will contain:
      Input:  uint8  [B, 3, H, W]  (raw RGB image, 0-255)
      Output: float32 [B, C, H, W] (per-pixel class scores, 0-100%)

    Baked-in operations:
      1. Cast uint8 → float32, divide by 255.0
      2. ImageNet mean/std normalization
      3. Forward through the segmentation backbone+decoder (logits)
      4. Softmax over the class dimension
      5. Multiply by 100.0 (percentage scale)
    """

    def __init__(self, seg_model, mean=None, std=None):
        super().__init__()
        self.seg_model = seg_model
        if mean is None:
            mean = [0.485, 0.456, 0.406]
        if std is None:
            std = [0.229, 0.224, 0.225]
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, x):
        # x: uint8 [B, 3, H, W] → float32 [B, 3, H, W] in [0, 1]
        x = x.float() / 255.0
        # ImageNet normalization
        x = (x - self.mean) / self.std
        # Segmentation model forward
        logits = self.seg_model(x)
        # Softmax over class dimension → probabilities [0, 1]
        probs = F.softmax(logits, dim=1)
        # Scale to 0-100%
        scores = probs * 100.0
        return scores


# ── Build the EBI wrapper and export ──────────────────
device = next(model.parameters()).device
model.eval()

ebi_model = EBIExportWrapper(model).to(device)
ebi_model.eval()

# Where to save ONNX models
onnx_dir = project_root / "models" / "onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / "unet_patches.onnx"

# Dummy input: uint8 image (as the ONNX graph expects raw images)
dummy_input = torch.randint(0, 256, (1, 3, 512, 512), dtype=torch.uint8, device=device)

torch.onnx.export(
    ebi_model,
    dummy_input,
    onnx_path.as_posix(),
    input_names=["input"],
    output_names=["output"],
    opset_version=12,
    dynamo=False,
    dynamic_axes={
        "input":  {0: "batch", 2: "height", 3: "width"},
        "output": {0: "batch", 2: "height", 3: "width"},
    },
)

print(f"✓ Exported EBI-compliant ONNX model to: {onnx_path}")
print(f"  Input:  uint8  [B, 3, H, W]  (raw RGB 0-255)")
print(f"  Output: float32 [B, C, H, W] (per-pixel class scores 0-100%)")

# ── Quick verification ────────────────────────────────
import onnxruntime as ort

session = ort.InferenceSession(onnx_path.as_posix())
inp_meta = session.get_inputs()[0]
out_meta = session.get_outputs()[0]
print(f"\nONNX Input:  name={inp_meta.name}, shape={inp_meta.shape}, type={inp_meta.type}")
print(f"ONNX Output: name={out_meta.name}, shape={out_meta.shape}, type={out_meta.type}")

test_input = np.random.randint(0, 256, (1, 3, 512, 512)).astype(np.uint8)
onnx_output = session.run(None, {inp_meta.name: test_input})[0]
print(f"  Output range: [{onnx_output.min():.2f}, {onnx_output.max():.2f}]")
print(f"  Sum along class axis: min={onnx_output.sum(axis=1).min():.2f}, max={onnx_output.sum(axis=1).max():.2f}")
print("✓ ONNX verification passed")

/tmp/ipykernel_60491/1962969584.py:68: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0318 17:34:44.072000 60491 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0318 17:34:45.214000 60491 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1

[torch.onnx] Obtain model graph for `EBIExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EBIExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/vscode/.local/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/vscode/.local/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/home/vscode/.local/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:65: adapter_lookup: Assertion `false` failed: No Adapter To Version $17 for Resize


Applied 94 of general pattern rewrite rules.
✓ Exported EBI-compliant ONNX model to: /workspaces/Oak-Defect-Detection/models/onnx/unet_patches.onnx
  Input:  uint8  [B, 3, H, W]  (raw RGB 0-255)
  Output: float32 [B, C, H, W] (per-pixel class scores 0-100%)

ONNX Input:  name=image, shape=['batch', 3, 'height', 'width'], type=tensor(uint8)
ONNX Output: name=scores, shape=['batch', 4, 'height', 'width'], type=tensor(float)
  Output range: [0.30, 96.61]
  Sum along class axis: min=100.00, max=100.00
✓ ONNX verification passed
